# Fase 3 - Camada Gold: perguntas de negócio e gráficos

Pipeline Medallion (Raw → Silver → **Gold**) sobre os dados de **Viagens a Serviço**
do Portal da Transparência (6 meses de 2025).

Este notebook responde às 7 perguntas de negócio do projeto:

| Parte | Pergunta | Fonte |
|---|---|---|
| A | 4. Tipo de pagamento com maior valor médio | SQL direto na Silver |
| A | 5. Meio de transporte mais usado nos trechos | SQL direto na Silver |
| A | 6. UF de destino em mais trechos | SQL direto na Silver |
| B | 1. 5 órgãos com maior custo total | tabela/VIEW Gold |
| B | 2. 3 destinos com maior custo médio por viagem | tabela/VIEW Gold |
| B | 3. Viagem de maior duração e seu custo total | tabela/VIEW Gold |
| B | 7. Órgão que pagou mais no total | tabela/VIEW Gold |

**Regras de negócio (definidas na Fase 2):**

- `valor_total = valor_diarias + valor_passagens + valor_outros_gastos - valor_devolucao`
- `duracao_dias = data_fim - data_inicio + 1` (conta o dia inicial e o final)
- Pagamentos e passagens 100% idênticos foram tratados como duplicidade e mantidos uma única vez.

**Pré-requisito:** rodar `0_criar_banco.sql`, `1_extrair.py` e `2_transformar.py`, nessa ordem.

## 1. Preparação: bibliotecas, conexão e funções auxiliares

In [ ]:
from decimal import Decimal

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from banco import conectar, executar

In [ ]:
# Cores do projeto (paleta validada para daltonismo e contraste).
COR_PRINCIPAL = "#2a78d6"   # azul: série principal
COR_SECUNDARIA = "#eb6834"  # laranja: segunda série
COR_TEXTO = "#0b0b0b"
COR_TEXTO_SUAVE = "#52514e"
COR_GRADE = "#e3e2de"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": COR_TEXTO_SUAVE,
    "axes.labelcolor": COR_TEXTO_SUAVE,
    "xtick.color": COR_TEXTO_SUAVE,
    "ytick.color": COR_TEXTO_SUAVE,
    "axes.titlecolor": COR_TEXTO,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.titlelocation": "left",
    "font.size": 10,
})

# Guarda a resposta de cada pergunta para o resumo final.
respostas = {}

In [ ]:
conexao = conectar()


def consultar(sql):
    """Executa um SELECT e devolve o resultado como DataFrame."""
    cursor = conexao.cursor()
    cursor.execute(sql)
    colunas = [coluna[0] for coluna in cursor.description]
    linhas = cursor.fetchall()
    cursor.close()
    tabela = pd.DataFrame(linhas, columns=colunas)
    for coluna in tabela.columns:
        if tabela[coluna].map(lambda v: isinstance(v, Decimal)).any():
            tabela[coluna] = tabela[coluna].astype(float)
    return tabela


def valor_unico(sql):
    """Executa um SELECT que devolve um único valor e retorna esse valor."""
    return consultar(sql).iloc[0, 0]


def moeda(valor):
    """Formata número como moeda brasileira: 1234.5 -> R$ 1.234,50."""
    texto = f"{valor:,.2f}".replace(",", "X").replace(".", ",")
    return "R$ " + texto.replace("X", ".")


def moeda_eixo(valor):
    """Formato curto para os eixos: 2500000 -> R$ 2,5 mi; 8000 -> R$ 8.000."""
    if abs(valor) >= 1_000_000:
        return "R$ " + f"{valor / 1_000_000:.1f}".replace(".", ",") + " mi"
    return "R$ " + inteiro(valor)


def inteiro(valor):
    """Formata inteiro com separador de milhar: 1234567 -> 1.234.567."""
    return f"{int(valor):,}".replace(",", ".")


def abreviar(texto, limite=38):
    """Encurta textos longos para caberem nos eixos dos gráficos."""
    texto = str(texto)
    return texto if len(texto) <= limite else texto[:limite - 3] + "..."


def mostrar(tabela, colunas_moeda=(), colunas_inteiro=()):
    """Exibe o DataFrame com valores formatados (sem alterar o original)."""
    copia = tabela.copy()
    for coluna in colunas_moeda:
        copia[coluna] = copia[coluna].map(moeda)
    for coluna in colunas_inteiro:
        copia[coluna] = copia[coluna].map(inteiro)
    display(copia.reset_index(drop=True))

In [ ]:
def barras_horizontais(rotulos, valores, titulo, eixo_x, eixo_y, formatar,
                       formatar_eixo=None):
    """
    Gráfico de barras horizontais (maior valor no topo) com o valor
    escrito na ponta de cada barra. Uma única série, uma única cor.
    """
    fig, eixo = plt.subplots(figsize=(9, 0.55 * len(rotulos) + 1.6))
    posicoes = range(len(rotulos))
    barras = eixo.barh(posicoes, valores, height=0.55, color=COR_PRINCIPAL)
    eixo.set_yticks(list(posicoes))
    eixo.set_yticklabels([abreviar(r) for r in rotulos])
    eixo.invert_yaxis()
    eixo.set_title(titulo)
    eixo.set_xlabel(eixo_x)
    eixo.set_ylabel(eixo_y)
    eixo.xaxis.grid(True, color=COR_GRADE, linewidth=0.8)
    eixo.set_axisbelow(True)
    formatar_eixo = formatar_eixo or formatar
    eixo.xaxis.set_major_formatter(lambda v, _: formatar_eixo(v))
    limite = max(valores) * 1.22
    eixo.set_xlim(0, limite)
    for barra, valor in zip(barras, valores):
        meio_barra = barra.get_y() + barra.get_height() / 2
        eixo.text(valor + limite * 0.01, meio_barra, formatar(valor),
                  va="center", color=COR_TEXTO, fontsize=9)
    fig.tight_layout()
    plt.show()

In [ ]:
# Teste rápido de conexão e do conteúdo da Silver.
contagens = consultar("""
    SELECT 'silver_viagem' AS tabela, COUNT(*) AS linhas FROM silver_viagem
    UNION ALL SELECT 'silver_pagamento', COUNT(*) FROM silver_pagamento
    UNION ALL SELECT 'silver_passagem', COUNT(*) FROM silver_passagem
    UNION ALL SELECT 'silver_trecho', COUNT(*) FROM silver_trecho
""")
mostrar(contagens, colunas_inteiro=["linhas"])

## 2. Parte A - Perguntas respondidas com SQL direto na Silver

Cada pergunta traz: **consulta SQL → tabela → gráfico → análise**.

### Pergunta 4 - Qual o tipo de pagamento com maior valor médio?

Consulta em `silver_pagamento`, agrupando por `tipo_pagamento`. A coluna `qtd_pagamentos`
mostra o tamanho de cada grupo, para não comparar médias de grupos muito pequenos sem perceber.

In [ ]:
SQL_PERGUNTA_4 = """
    SELECT tipo_pagamento,
           COUNT(*)              AS qtd_pagamentos,
           ROUND(AVG(valor), 2)  AS valor_medio,
           ROUND(SUM(valor), 2)  AS valor_total
    FROM silver_pagamento
    WHERE valor IS NOT NULL
    GROUP BY tipo_pagamento
    ORDER BY valor_medio DESC
"""
p4 = consultar(SQL_PERGUNTA_4)
mostrar(p4, ["valor_medio", "valor_total"], ["qtd_pagamentos"])

In [ ]:
barras_horizontais(
    p4["tipo_pagamento"], p4["valor_medio"],
    titulo="Valor médio por tipo de pagamento",
    eixo_x="Valor médio por pagamento (R$)",
    eixo_y="Tipo de pagamento",
    formatar=moeda,
    formatar_eixo=moeda_eixo,
)

lider = p4.iloc[0]
media_geral = (p4["valor_total"].sum() / p4["qtd_pagamentos"].sum())
insight = (
    f"O tipo '{lider['tipo_pagamento']}' tem o maior valor médio: "
    f"{moeda(lider['valor_medio'])} por pagamento "
    f"({lider['valor_medio'] / media_geral:.1f}x a média geral de "
    f"{moeda(media_geral)}), em {inteiro(lider['qtd_pagamentos'])} pagamentos."
)
respostas["Pergunta 4"] = insight
print("Análise:", insight)

### Pergunta 5 - Qual o meio de transporte mais usado nos trechos?

Consulta em `silver_trecho`, contando os trechos por `meio_transporte`.
Trechos sem o meio informado entram como "Não informado".

In [ ]:
SQL_PERGUNTA_5 = """
    SELECT COALESCE(meio_transporte, 'Não informado') AS meio_transporte,
           COUNT(*) AS qtd_trechos,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentual
    FROM silver_trecho
    GROUP BY COALESCE(meio_transporte, 'Não informado')
    ORDER BY qtd_trechos DESC
"""
p5 = consultar(SQL_PERGUNTA_5)
mostrar(p5, colunas_inteiro=["qtd_trechos"])

In [ ]:
barras_horizontais(
    p5["meio_transporte"], p5["qtd_trechos"],
    titulo="Trechos por meio de transporte",
    eixo_x="Quantidade de trechos",
    eixo_y="Meio de transporte",
    formatar=inteiro,
)

lider = p5.iloc[0]
insight = (
    f"'{lider['meio_transporte']}' é o meio mais usado: "
    f"{inteiro(lider['qtd_trechos'])} trechos, "
    f"{lider['percentual']:.1f}% do total."
)
if len(p5) > 1:
    segundo = p5.iloc[1]
    insight += (
        f" O segundo colocado é '{segundo['meio_transporte']}', com "
        f"{segundo['percentual']:.1f}%."
    )
respostas["Pergunta 5"] = insight
print("Análise:", insight)

### Pergunta 6 - Qual UF de destino aparece em mais trechos?

Consulta em `silver_trecho`, contando os trechos por `destino_uf`.

Como a volta para a cidade de origem também é um trecho, a UF-base dos viajantes tende a
liderar a contagem. Por isso o gráfico compara **todos os trechos** com **apenas o 1º trecho
de cada viagem** (a ida), que mostra para onde as pessoas realmente vão.

In [ ]:
SQL_PERGUNTA_6_TODOS = """
    SELECT destino_uf AS uf, COUNT(*) AS qtd_trechos
    FROM silver_trecho
    WHERE destino_uf IS NOT NULL
    GROUP BY destino_uf
    ORDER BY qtd_trechos DESC
"""
SQL_PERGUNTA_6_IDA = """
    SELECT destino_uf AS uf, COUNT(*) AS qtd_trechos_ida
    FROM silver_trecho
    WHERE destino_uf IS NOT NULL AND sequencia_trecho = 1
    GROUP BY destino_uf
"""
todos = consultar(SQL_PERGUNTA_6_TODOS)
ida = consultar(SQL_PERGUNTA_6_IDA)
p6 = todos.head(10).merge(ida, on="uf", how="left").fillna(0)
p6["qtd_trechos_ida"] = p6["qtd_trechos_ida"].astype(int)
mostrar(p6, colunas_inteiro=["qtd_trechos", "qtd_trechos_ida"])

In [ ]:
fig, eixo = plt.subplots(figsize=(10, 5))
largura = 0.38
x = range(len(p6))
eixo.bar([i - largura / 2 for i in x], p6["qtd_trechos"], width=largura,
         color=COR_PRINCIPAL, label="Todos os trechos")
eixo.bar([i + largura / 2 for i in x], p6["qtd_trechos_ida"], width=largura,
         color=COR_SECUNDARIA, label="Apenas o 1º trecho (ida)")
eixo.set_xticks(list(x))
eixo.set_xticklabels(p6["uf"])
eixo.set_title("UFs de destino com mais trechos (top 10)")
eixo.set_xlabel("UF de destino")
eixo.set_ylabel("Quantidade de trechos")
eixo.yaxis.set_major_formatter(lambda v, _: inteiro(v))
eixo.yaxis.grid(True, color=COR_GRADE, linewidth=0.8)
eixo.set_axisbelow(True)
eixo.legend(frameon=False, loc="upper right")
fig.tight_layout()
plt.show()

lider = p6.iloc[0]
pct = 100 * lider["qtd_trechos"] / todos["qtd_trechos"].sum()
insight = (
    f"A UF de destino em mais trechos é {lider['uf']}, com "
    f"{inteiro(lider['qtd_trechos'])} trechos ({pct:.1f}% do total)."
)
lider_ida = ida.sort_values("qtd_trechos_ida", ascending=False).iloc[0]
if lider_ida["uf"] != lider["uf"]:
    insight += (
        f" Considerando só a ida (1º trecho), a UF líder muda para "
        f"{lider_ida['uf']} ({inteiro(lider_ida['qtd_trechos_ida'])} trechos)."
    )
else:
    insight += (
        f" Considerando só a ida (1º trecho), {lider_ida['uf']} continua "
        f"líder, com {inteiro(lider_ida['qtd_trechos_ida'])} trechos."
    )
respostas["Pergunta 6"] = insight
print("Análise:", insight)